In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv
/kaggle/input/datasets/kgan31/doha-context/dohas_nlp_ready_simple_lang.csv


In [13]:
!pip install transformers torch sentencepiece
CSV_PATH = "/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv"

In [9]:
"""
GPT2-Hindi Zero-Shot Doha Generation
=====================================
Generates dohas given theme and context WITHOUT any fine-tuning.
Uses surajp/gpt2-hindi — a causal LM pretrained on Hindi text.

This serves as your zero-shot baseline before fine-tuning.

Kaggle setup:
    !pip install transformers torch sentencepiece
    CSV_PATH = "/kaggle/input/doha-dataset/doha_dataset.csv"

Local setup:
    pip install transformers torch sentencepiece pandas
"""

import re
import random
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

MODEL_NAME  = "surajp/gpt2-hindi"
# CSV_PATH    = "/kaggle/input/datasets/kgan31/doha-context/dohas_nlp_ready_simple_lang.csv"    # ← change to your path

# Generation settings
MAX_NEW_TOKENS  = 100
TEMPERATURE     = 0.8
TOP_P           = 0.92              # nucleus sampling — keeps top 92% probability mass
TOP_K           = 50                # also limits to top-50 tokens at each step
NUM_RETURN      = 3                 # generate 3 candidates per prompt
REPETITION_PEN  = 1.3              # penalise repeated tokens

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42
random.seed(SEED)
torch.manual_seed(SEED)


# ─────────────────────────────────────────────────────────────────────────────
# PROMPT TEMPLATES
# ─────────────────────────────────────────────────────────────────────────────
# GPT2 is a causal LM — it continues whatever text you give it.
# The prompt structure heavily influences output quality.
# We try 3 templates and let you pick the best one.

def make_prompt_v1(theme: str, context: str) -> str:
    """
    Template 1 — Direct doha instruction.
    Simple and clean. Works best when the model has seen doha-like text.
    """
    return (
        f"विषय: {theme}\n"
        f"संदर्भ: {context}\n"
        f"दोहा:\n"
    )


def make_prompt_v2(theme: str, context: str) -> str:
    """
    Template 2 — Few-shot style with a doha example prefix.
    Giving one example doha in the prompt helps GPT2 understand the format.
    """
    return (
        f"नीचे दिए गए विषय और संदर्भ पर एक दोहा लिखिए।\n\n"
        f"विषय: शृंगार\n"
        f"संदर्भ: नायिका के नेत्र\n"
        f"दोहा: नैन नचाय कह गई, बहुत दिनन के बाद।\n"
        f"अब आइयो राधा प्यारी, सुनिए श्याम की बात।।\n\n"
        f"विषय: {theme}\n"
        f"संदर्भ: {context}\n"
        f"दोहा:"
    )


def make_prompt_v3(theme: str, context: str) -> str:
    """
    Template 3 — Minimal, just the context as a seed.
    Sometimes less instruction = more natural generation.
    """
    return f"{context} पर दोहा — {theme}:\n"


# ─────────────────────────────────────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────────────────────────────────────

def load_model(model_name: str = MODEL_NAME):
    """Load GPT2-Hindi tokenizer and model."""
    print(f"⬇  Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    total = sum(p.numel() for p in model.parameters())
    print(f"✅ Loaded — {total:,} parameters | Device: {DEVICE}\n")
    return tokenizer, model


# ─────────────────────────────────────────────────────────────────────────────
# GENERATION
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def generate_doha_zeroshot(
    tokenizer,
    model,
    theme: str,
    context: str,
    prompt_fn=make_prompt_v2,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    top_k: int = TOP_K,
    num_return: int = NUM_RETURN,
    repetition_penalty: float = REPETITION_PEN,
) -> list:
    """
    Generate doha candidates zero-shot given theme and context.

    Uses nucleus sampling (top_p) + top_k for diverse, quality outputs.
    Returns num_return candidates so you can pick the best.

    Args:
        theme    : Hindi theme  e.g. "विरह"
        context  : Hindi context e.g. "विरह की अग्नि"
        prompt_fn: Which prompt template to use (v1, v2, or v3)

    Returns:
        List of generated strings (prompt stripped out)
    """
    prompt = prompt_fn(theme.strip(), context.strip())
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_len = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        num_return_sequences=num_return,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,  # suppress pad warning
    )

    results = []
    for out in outputs:
        # Decode only the newly generated tokens (strip prompt)
        generated_ids = out[input_len:]
        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        text = clean_output(text)
        results.append(text)

    return results


def clean_output(text: str) -> str:
    """
    Clean up generated text:
      - Strip leading/trailing whitespace
      - Cut off after second danda (।।) — doha is 2 lines
      - Remove any trailing incomplete lines
    """
    text = text.strip()

    # Try to extract just 2 lines (one complete doha)
    # A doha ends with ।। so cut after second occurrence
    parts = text.split("।।")
    if len(parts) >= 2:
        text = parts[0].strip() + " ।।\n" + parts[1].strip() + " ।।"
    else:
        # Fallback: take first 2 newline-separated lines
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        text  = "\n".join(lines[:2]) if lines else text

    return text.strip()


# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION (zero-shot BLEU)
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_zeroshot_bleu(tokenizer, model, df: pd.DataFrame,
                           n_samples: int = 20) -> dict:
    """
    Compute character-level BLEU for zero-shot generation on n_samples rows.
    Uses the best candidate (highest BLEU-1) among num_return generations.
    """
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    import numpy as np
    nltk_ok = True
    try:
        import nltk
        nltk.download("punkt", quiet=True)
    except Exception:
        nltk_ok = False

    smoother  = SmoothingFunction().method1
    all_dohas = df["Doha"].apply(lambda x: re.sub(r"।।\s*\d+\s*।।", "।।",
                                                   str(x))).tolist()
    sample_df = df.sample(min(n_samples, len(df)), random_state=SEED)

    all_b1, all_b2, all_b4 = [], [], []

    print(f"📊 Zero-shot BLEU on {len(sample_df)} samples...")
    for _, row in sample_df.iterrows():
        candidates = generate_doha_zeroshot(
            tokenizer, model,
            str(row["Theme"]), str(row["Context"]),
            num_return=3
        )

        # Pick candidate with highest BLEU-1 vs reference corpus
        best_score, best_b2, best_b4 = 0, 0, 0
        for cand in candidates:
            hyp = list(cand)
            refs = [list(r) for r in all_dohas]
            b1 = sentence_bleu(refs, hyp,
                                weights=(1,0,0,0),
                                smoothing_function=smoother)
            b2 = sentence_bleu(refs, hyp,
                                weights=(0.5,0.5,0,0),
                                smoothing_function=smoother)
            b4 = sentence_bleu(refs, hyp,
                                weights=(0.25,0.25,0.25,0.25),
                                smoothing_function=smoother)
            if b1 > best_score:
                best_score, best_b2, best_b4 = b1, b2, b4

        all_b1.append(best_score)
        all_b2.append(best_b2)
        all_b4.append(best_b4)

    import numpy as np
    results = {
        "bleu1": np.mean(all_b1),
        "bleu2": np.mean(all_b2),
        "bleu4": np.mean(all_b4),
    }
    print(f"   BLEU-1 : {results['bleu1']:.4f}")
    print(f"   BLEU-2 : {results['bleu2']:.4f}")
    print(f"   BLEU-4 : {results['bleu4']:.4f}")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# MAIN — DEMO
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print(f"\n{'='*60}")
    print("GPT2-Hindi — Zero-Shot Doha Generation")
    print(f"{'='*60}\n")

    tokenizer, model = load_model()

    # ── Load CSV to get real theme/context pairs ───────────
    df = pd.read_csv(CSV_PATH)
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=["Doha", "Theme", "Context"]).reset_index(drop=True)
    print(f"✅ Loaded {len(df)} dohas from CSV\n")

    # ── Demo: generate for 5 real pairs ───────────────────
    print("=" * 60)
    print("ZERO-SHOT GENERATION — 3 prompts × 3 candidates each")
    print("=" * 60)

    sample_rows = df.sample(5, random_state=SEED)

    for _, row in sample_rows.iterrows():
        theme   = str(row["Theme"])
        context = str(row["Context"])
        ref     = str(row["Doha"])[:80]

        print(f"\n  Theme   : {theme}")
        print(f"  Context : {context}")
        print(f"  Reference (first 80 chars): {ref}")
        print()

        for i, prompt_fn in enumerate([make_prompt_v1, make_prompt_v2, make_prompt_v3], 1):
            candidates = generate_doha_zeroshot(
                tokenizer, model, theme, context,
                prompt_fn=prompt_fn, num_return=3
            )
            print(f"  Template {i}:")
            for j, cand in enumerate(candidates, 1):
                print(f"    Candidate {j}: {cand[:100]}")
        print("-" * 60)

    # ── BLEU baseline evaluation ───────────────────────────
    print(f"\n{'='*60}")
    print("ZERO-SHOT BLEU BASELINE")
    print(f"{'='*60}")
    evaluate_zeroshot_bleu(tokenizer, model, df, n_samples=20)

    # ── Interactive mode ───────────────────────────────────
    print(f"\n{'='*60}")
    print("INTERACTIVE MODE (type 'quit' to exit)")
    print(f"{'='*60}")
    while True:
        theme   = input("\nEnter theme (विषय)   : ").strip()
        if theme.lower() in ("quit", "exit", "q"):
            break
        context = input("Enter context (संदर्भ): ").strip()
        if not context:
            continue

        print("\nGenerating with Template 2 (few-shot)...")
        candidates = generate_doha_zeroshot(
            tokenizer, model, theme, context,
            prompt_fn=make_prompt_v2, num_return=3
        )
        for i, cand in enumerate(candidates, 1):
            print(f"  Candidate {i}:\n    {cand}\n")


if __name__ == "__main__":
    main()


GPT2-Hindi — Zero-Shot Doha Generation

⬇  Loading surajp/gpt2-hindi...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: surajp/gpt2-hindi
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded — 81,912,576 parameters | Device: cuda

✅ Loaded 7889 dohas from CSV

ZERO-SHOT GENERATION — 3 prompts × 3 candidates each

  Theme   : जीवन
  Context : आखिरी सहारा
  Reference (first 80 chars): खींच लिया जल सतह का, सूख गये सब कूप
इस छोटे से गाँव के, तुम्हीं प्रमुख नलकूप ॥

  Template 1:
    Candidate 1: लेकिन यहां यह विश्वसनीय नहीं है।
पुलिसकर्मियों को बचाने में खूब सावधान रहना।
    Candidate 2: और बात कुछ और है , नौकरी करने वालों में यूनुसजी ज़्यादा शांत नहीं हैं .
जिससे पंचायत भाजपा के खिलाफ 
    Candidate 3: के माध्यम से,
इस तरह हमें इस सुख की प्रकृति को ही अपनाना चाहिए।
  Template 2:
    Candidate 1: रामपुरम, रामसिंघ में डॉ.
लोकतांत्रिक चेतना में बदलाव आया।
    Candidate 2: इस मूल कथा-में बांध बनाते-बांध बनते, तो यहां पर हिंसा है।
एक और उद्धरण लिखा, 'यह दोनों में खासतौर पर
    Candidate 3: मैं तो अपने चूतड़ पकड़ा नहीं है।
इस दौरान उन्होंने टॉवर पर जमकर धारदार हथियार से हमला कर उसे घायल कर
  Template 3:
    Candidate 1: फ़िल्म के पहले रोमांस और नॉक आउट हुए फ़्लैश और लंबे ब

KeyboardInterrupt: Interrupted by user


Enter theme (विषय)   :  10.


In [14]:
"""
GPT2-Hindi Fine-tuning for Hindi Doha Generation
=================================================
Fine-tunes surajp/gpt2-hindi on (Theme + Context → Doha) pairs.

GPT2 is a causal LM — it learns to continue a prompt.
Training format (one sequence per doha):

    विषय: शृंगार\nसंदर्भ: मोरपंखी बाल\nदोहा:\nमोर पच्छ जो सिर चढ़ै...<|endoftext|>

At inference, we feed everything up to "दोहा:\n" and let the model
complete the rest — giving us conditioned doha generation.

Why GPT2 over IndicBART for this task:
  - Causal LM fine-tuning is simpler and more stable on small data
  - No encoder/decoder mismatch issues
  - No language token complications
  - Braj Bhasha patterns are learned directly from your data

Kaggle setup:
    !pip install transformers torch sentencepiece tqdm
    CSV_PATH = "/kaggle/input/doha-dataset/doha_dataset.csv"
    SAVE_DIR = "/kaggle/working"

Local setup:
    pip install transformers torch sentencepiece tqdm pandas numpy nltk
"""

import os
import re
import math
import random
import time
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download("punkt", quiet=True)

from tqdm.auto import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

CSV_PATH      = "/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv"       # ← change to your path
SAVE_DIR      = "/kaggle/working/"                      # ← where to save checkpoints

MODEL_NAME    = "surajp/gpt2-hindi"

# Training
EPOCHS        = 15
BATCH_SIZE    = 16
GRAD_ACCUM    = 2                        # effective batch = 32
LR            = 3e-5                     # slightly lower than IndicBART
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
MAX_GRAD_NORM = 1.0
VAL_SPLIT     = 0.1
PATIENCE      = 4

# Tokenization
MAX_SEQ_LEN   = 192                      # prompt + doha fits in 192 tokens

# Generation
MAX_NEW_TOKENS    = 100
TEMPERATURE       = 0.8
TOP_P             = 0.92
TOP_K             = 50
REPETITION_PEN    = 1.3
NUM_BEAMS         = 1                    # sampling is better than beam for GPT2

SEED   = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

# Separator between prompt and doha body — used to split during generation
DOHA_SEP = "दोहा:\n"


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"।।\s*\d+\s*।।", "।।", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def make_training_sequence(theme: str, context: str, doha: str,
                           eos_token: str) -> str:
    """
    Build one full training sequence for causal LM fine-tuning.

    Format:
        विषय: <theme>
        संदर्भ: <context>
        दोहा:
        <doha><EOS>

    The model learns to predict every token including the prompt,
    but we only compute loss on the doha part (see DohaDataset).
    """
    return (
        f"विषय: {clean_text(theme)}\n"
        f"संदर्भ: {clean_text(context)}\n"
        f"दोहा:\n"
        f"{clean_text(doha)}{eos_token}"
    )


def make_inference_prompt(theme: str, context: str) -> str:
    """
    Build the prompt for inference — everything except the doha body.
    The model will complete this prompt with a generated doha.
    """
    return (
        f"विषय: {clean_text(theme)}\n"
        f"संदर्भ: {clean_text(context)}\n"
        f"दोहा:\n"
    )


def load_dataset(csv_path: str, eos_token: str):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    required = {"Doha", "Theme", "Context"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")

    df = df.dropna(subset=["Doha", "Theme", "Context"]).reset_index(drop=True)

    sequences = []
    for _, row in df.iterrows():
        seq = make_training_sequence(
            str(row["Theme"]), str(row["Context"]),
            str(row["Doha"]), eos_token
        )
        sequences.append(seq)

    print(f"✅ Loaded {len(sequences)} sequences")
    print(f"\n   Sample sequence:\n   {sequences[0]}\n")
    return sequences, df


def split_data(sequences, df, val_split):
    indices = list(range(len(sequences)))
    random.shuffle(indices)
    split       = int(len(indices) * (1 - val_split))
    train_idx   = indices[:split]
    val_idx     = indices[split:]
    train_seqs  = [sequences[i] for i in train_idx]
    val_seqs    = [sequences[i] for i in val_idx]
    val_df      = df.iloc[val_idx].reset_index(drop=True)
    return train_seqs, val_seqs, val_df


# ─────────────────────────────────────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────────────────────────────────────

class DohaDataset(Dataset):
    """
    Tokenizes full training sequences for causal LM training.

    Key design: we only compute loss on the DOHA part, not the prompt.
    This is done by setting labels = -100 for all prompt tokens.

    Why? We don't want the model to "learn" to generate the prompt —
    we want it to learn to generate the doha given the prompt.

    Example:
        Full sequence : "विषय: शृंगार\\nसंदर्भ: ...\\nदोहा:\\nमोर पच्छ..."
        Labels        :  -100 -100 ...  -100 -100 ... मोर  पच्छ  ...
                         ← prompt tokens ignored →    ← loss computed here →
    """

    def __init__(self, sequences: list, tokenizer, max_seq_len: int):
        self.tokenizer   = tokenizer
        self.max_seq_len = max_seq_len
        self.samples     = []

        # Pre-tokenize the prompt separator to find where doha starts
        sep_ids = tokenizer.encode(DOHA_SEP, add_special_tokens=False)

        for seq in sequences:
            enc = tokenizer(
                seq,
                max_length=max_seq_len,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )
            input_ids      = enc["input_ids"].squeeze()
            attention_mask = enc["attention_mask"].squeeze()

            # Build labels: -100 for prompt, real ids for doha
            labels = input_ids.clone()

            # Find where the doha starts by locating DOHA_SEP in the sequence
            ids_list = input_ids.tolist()
            sep_start = self._find_sublist(ids_list, sep_ids)

            if sep_start != -1:
                # Mask everything up to and including the separator
                doha_start = sep_start + len(sep_ids)
                labels[:doha_start] = -100
            else:
                # Fallback: mask first half if separator not found
                labels[:len(ids_list)//2] = -100

            # Also mask padding
            labels[attention_mask == 0] = -100

            self.samples.append({
                "input_ids":      input_ids,
                "attention_mask": attention_mask,
                "labels":         labels,
            })

    @staticmethod
    def _find_sublist(lst: list, sub: list) -> int:
        """Find starting index of sub in lst, return -1 if not found."""
        n, m = len(lst), len(sub)
        for i in range(n - m + 1):
            if lst[i:i+m] == sub:
                return i
        return -1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ─────────────────────────────────────────────────────────────────────────────
# 3. TRAINING
# ─────────────────────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, device,
                grad_accum, max_grad_norm, epoch, total_epochs):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    bar = tqdm(
        loader,
        desc=f"Epoch {epoch:>2}/{total_epochs} [train]",
        unit="batch",
        leave=False,
        dynamic_ncols=True,
    )

    for step, batch in enumerate(bar):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += outputs.loss.item()
        bar.set_postfix(loss=f"{total_loss / (step + 1):.4f}")

    bar.close()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loss(model, loader, device, epoch, total_epochs):
    model.eval()
    total_loss = 0.0

    bar = tqdm(
        loader,
        desc=f"Epoch {epoch:>2}/{total_epochs} [ val ]",
        unit="batch",
        leave=False,
        dynamic_ncols=True,
    )

    for step, batch in enumerate(bar):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        total_loss += outputs.loss.item()
        bar.set_postfix(loss=f"{total_loss / (step + 1):.4f}")

    bar.close()
    return total_loss / len(loader)


# ─────────────────────────────────────────────────────────────────────────────
# 4. GENERATION
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def generate_doha(model, tokenizer, theme: str, context: str,
                  max_new_tokens: int = MAX_NEW_TOKENS,
                  temperature: float = TEMPERATURE,
                  top_p: float = TOP_P,
                  top_k: int = TOP_K,
                  repetition_penalty: float = REPETITION_PEN,
                  device: str = DEVICE) -> str:
    """
    Generate a doha given theme and context.

    Feeds the prompt up to "दोहा:\\n" and lets the model
    autoregressively complete the rest.
    """
    model.eval()
    prompt = make_inference_prompt(theme, context)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode only newly generated tokens
    generated_ids = outputs[0][prompt_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return clean_output(text)


def clean_output(text: str) -> str:
    """Extract clean doha — cut after second danda ।।"""
    text  = text.strip()
    parts = text.split("।।")
    if len(parts) >= 2:
        line1 = parts[0].strip()
        line2 = parts[1].strip()
        return f"{line1} ।।\n{line2} ।।"
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return "\n".join(lines[:2]) if lines else text


# ─────────────────────────────────────────────────────────────────────────────
# 5. EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def compute_perplexity(avg_loss: float) -> float:
    return math.exp(min(avg_loss, 500))


def run_bleu_evaluation(model, tokenizer, val_df: pd.DataFrame,
                        all_dohas: list, n_samples: int = 20,
                        device: str = DEVICE) -> dict:
    smoother = SmoothingFunction().method1
    sample_df = val_df.sample(min(n_samples, len(val_df)), random_state=SEED)
    all_b1, all_b2, all_b4 = [], [], []

    bar = tqdm(sample_df.iterrows(), total=len(sample_df),
               desc="  BLEU eval", unit="sample", dynamic_ncols=True)

    for _, row in bar:
        gen       = generate_doha(model, tokenizer,
                                  str(row["Theme"]), str(row["Context"]),
                                  device=device)
        hyp_chars = list(gen)
        ref_chars = [list(r) for r in all_dohas]

        b1 = sentence_bleu(ref_chars, hyp_chars, weights=(1,0,0,0),
                           smoothing_function=smoother)
        b2 = sentence_bleu(ref_chars, hyp_chars, weights=(0.5,0.5,0,0),
                           smoothing_function=smoother)
        b4 = sentence_bleu(ref_chars, hyp_chars,
                           weights=(0.25,0.25,0.25,0.25),
                           smoothing_function=smoother)
        all_b1.append(b1)
        all_b2.append(b2)
        all_b4.append(b4)
        bar.set_postfix(bleu1=f"{np.mean(all_b1):.4f}")

    bar.close()
    results = {
        "bleu1": np.mean(all_b1),
        "bleu2": np.mean(all_b2),
        "bleu4": np.mean(all_b4),
    }
    print(f"   BLEU-1 : {results['bleu1']:.4f}")
    print(f"   BLEU-2 : {results['bleu2']:.4f}")
    print(f"   BLEU-4 : {results['bleu4']:.4f}")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# 6. MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print(f"\n{'='*60}")
    print("GPT2-Hindi Fine-tuning — Hindi Doha Generation")
    print(f"Model  : {MODEL_NAME}")
    print(f"Device : {DEVICE}")
    print(f"{'='*60}\n")

    # ── Load tokenizer & model ─────────────────────────────
    print(f"⬇  Loading {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # GPT2 has no pad token by default — set it to eos
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.resize_token_embeddings(len(tokenizer))   # in case vocab changed

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Total params     : {total:,}")
    print(f"   Trainable params : {trainable:,}\n")

    # ── Load data ──────────────────────────────────────────
    sequences, df = load_dataset(CSV_PATH, tokenizer.eos_token)
    train_seqs, val_seqs, val_df = split_data(sequences, df, VAL_SPLIT)
    all_dohas = df["Doha"].apply(clean_text).tolist()

    print(f"   Train sequences : {len(train_seqs)}")
    print(f"   Val sequences   : {len(val_seqs)}\n")

    # ── Build datasets ─────────────────────────────────────
    print("⚙  Tokenizing dataset (pre-tokenizing for speed)...")
    train_dataset = DohaDataset(train_seqs, tokenizer, MAX_SEQ_LEN)
    val_dataset   = DohaDataset(val_seqs,   tokenizer, MAX_SEQ_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)
    print(f"   Train batches : {len(train_loader)}")
    print(f"   Val batches   : {len(val_loader)}\n")

    # ── Optimizer & scheduler ─────────────────────────────
    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    print(f"   Total steps  : {total_steps}")
    print(f"   Warmup steps : {warmup_steps}\n")

    # ── Training ───────────────────────────────────────────
    epoch_bar = tqdm(
        range(1, EPOCHS + 1),
        desc="Overall progress",
        unit="epoch",
        dynamic_ncols=True,
    )

    best_val_loss      = float("inf")
    early_stop_counter = 0
    history            = []

    tqdm.write(
        f"\n{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} "
        f"{'Train PPL':>11} {'Val PPL':>9} {'LR':>10} {'Time':>7}"
    )
    tqdm.write("-" * 68)

    for epoch in epoch_bar:
        t0 = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, scheduler,
                                 DEVICE, GRAD_ACCUM, MAX_GRAD_NORM,
                                 epoch, EPOCHS)
        val_loss   = evaluate_loss(model, val_loader, DEVICE, epoch, EPOCHS)

        train_ppl  = compute_perplexity(train_loss)
        val_ppl    = compute_perplexity(val_loss)
        elapsed    = time.time() - t0
        current_lr = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
            "train_ppl": train_ppl, "val_ppl": val_ppl, "lr": current_lr
        })

        epoch_bar.set_postfix(
            train_loss=f"{train_loss:.4f}",
            val_ppl=f"{val_ppl:.2f}",
        )

        tqdm.write(
            f"{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f} "
            f"{train_ppl:>11.2f} {val_ppl:>9.2f} {current_lr:>10.2e} {elapsed:>6.1f}s"
        )

        # Save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stop_counter = 0
            ckpt_dir = os.path.join(SAVE_DIR, "best_gpt2hindi_doha")
            os.makedirs(ckpt_dir, exist_ok=True)
            model.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
            tqdm.write(f"         💾 Saved best model (val_ppl={val_ppl:.2f})")
        else:
            early_stop_counter += 1
            if early_stop_counter >= PATIENCE:
                tqdm.write(
                    f"\n⏹  Early stopping at epoch {epoch} "
                    f"(no improvement for {PATIENCE} epochs)"
                )
                break

        # Sample every 3 epochs
        if epoch % 3 == 0 and len(val_df) > 0:
            row = val_df.iloc[0]
            gen = generate_doha(model, tokenizer,
                                str(row["Theme"]), str(row["Context"]))
            tqdm.write(f"\n  🎵 Theme={row['Theme']!r} | Context={row['Context']!r}")
            tqdm.write(f"     Generated → {gen}")
            tqdm.write(f"     Reference → {clean_text(str(row['Doha']))[:80]}\n")

    epoch_bar.close()

    # ── Final eval ─────────────────────────────────────────
    print(f"\n{'='*60}")
    print("FINAL EVALUATION")
    print(f"{'='*60}\n")

    best_model = AutoModelForCausalLM.from_pretrained(
        os.path.join(SAVE_DIR, "best_gpt2hindi_doha")
    ).to(DEVICE)
    best_tok   = AutoTokenizer.from_pretrained(
        os.path.join(SAVE_DIR, "best_gpt2hindi_doha")
    )
    if best_tok.pad_token is None:
        best_tok.pad_token = best_tok.eos_token

    bleu_scores = run_bleu_evaluation(
        best_model, best_tok, val_df,
        all_dohas, n_samples=20, device=DEVICE
    )

    pd.DataFrame(history).to_csv(
        os.path.join(SAVE_DIR, "gpt2hindi_history.csv"), index=False
    )
    print(f"\n  📈 History saved → {SAVE_DIR}/gpt2hindi_history.csv")

    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"  Model        : {MODEL_NAME}")
    print(f"  Best Val PPL : {compute_perplexity(best_val_loss):.2f}")
    print(f"  BLEU-1       : {bleu_scores['bleu1']:.4f}")
    print(f"  BLEU-2       : {bleu_scores['bleu2']:.4f}")
    print(f"  BLEU-4       : {bleu_scores['bleu4']:.4f}")
    print(f"{'='*60}\n")

    # Final samples
    print("GENERATED DOHAS")
    print("-" * 55)
    for _, row in val_df.sample(min(5, len(val_df)), random_state=SEED).iterrows():
        gen = generate_doha(best_model, best_tok,
                            str(row["Theme"]), str(row["Context"]))
        print(f"  Theme     : {row['Theme']}")
        print(f"  Context   : {row['Context']}")
        print(f"  Generated → {gen}")
        print(f"  Reference → {clean_text(str(row['Doha']))[:100]}")
        print("-" * 55)


if __name__ == "__main__":
    main()


GPT2-Hindi Fine-tuning — Hindi Doha Generation
Model  : surajp/gpt2-hindi
Device : cuda

⬇  Loading surajp/gpt2-hindi...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: surajp/gpt2-hindi
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Total params     : 81,913,344
   Trainable params : 81,913,344

✅ Loaded 7889 sequences

   Sample sequence:
   विषय: शृंगार
संदर्भ: मोरपंखी बाल
दोहा:
मोर पच्छ जो सिर चढ़ै बारन तें अधिकाय। सहस चखन लखि धनि कचन परे मान छिन पाय ॥<|endoftext|>

   Train sequences : 7100
   Val sequences   : 789

⚙  Tokenizing dataset (pre-tokenizing for speed)...
   Train batches : 444
   Val batches   : 50

   Total steps  : 3330
   Warmup steps : 333



Overall progress:   0%|          | 0/15 [00:00<?, ?epoch/s]

...

Epoch  9/15 [ val ]:   0%|          | 0/50 [00:00<?, ?batch/s]

     9       2.0540     2.5242        7.80     12.48   1.33e-05  233.3s

⏹  Early stopping at epoch 9 (no improvement for 4 epochs)

FINAL EVALUATION



Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

  BLEU eval:   0%|          | 0/20 [00:00<?, ?sample/s]

   BLEU-1 : 0.9214
   BLEU-2 : 0.8829
   BLEU-4 : 0.8177

  📈 History saved → /kaggle/working//gpt2hindi_history.csv

SUMMARY
  Model        : surajp/gpt2-hindi
  Best Val PPL : 12.27
  BLEU-1       : 0.9214
  BLEU-2       : 0.8829
  BLEU-4       : 0.8177

GENERATED DOHAS
-------------------------------------------------------
  Theme     : नीति
  Context   : समय के साथ चलो
  Generated → समय की बर्बादी, कर दें पुराने कौन? मूल मंजूर है वक़्त तो, जब लग जाएँगे शौक ॥ to all the way, it should have been true. ॥ a newest posts with label and there is not in your friend, as their mind for how! ॥'s blog parichre meed off; ॥-in
  Reference → अपने मत धूनल करीं, चलीं समय के साथ। बेटो से पूछल करीं, ओकरा मन के बात ॥
-------------------------------------------------------
  Theme     : जीवन
  Context   : जीना मुश्किल है
  Generated → तेरा कौशल सँग, जीवन-जीना मुश्किल । अब तू मैं रहूँ, कृष्णा नाथ के लोट ॥, बजे भी तो हैं, पाँच हज़ार ॥, और श्याम सखी, मोहन दानि पुलकित ॥) आ पहुँचते, पहुँचा,
  Reference → 